# Study 1: Leakage-Safe GI VQA Benchmark and Visual Faithfulness Audit

First-paper pipeline for a reproducible GI visual question answering benchmark and causal audit of visual attribution methods.

**Study aim:** measure how fine-tuning changes GI-VQA performance and test whether answer-token-specific visual attributions identify image evidence that causally affects model outputs.

**Core hypothesis:** if a visual attribution is faithful, deleting its most salient regions should reduce the fixed saved answer's mean token log probability more than matched random and least-salient deletion, while inserting those regions should restore model support faster than the controls.

**Paper boundary:** this study evaluates answer performance, visual-attribution faithfulness, and model confidence. It does not claim that an attribution is clinically plausible, that confidence is calibrated without testing it, or that an explanation improves clinician trust or decisions.

## Study Design

### Research Questions

1. **VQA performance:** How much does fine-tuning improve GI-VQA performance across question classes and complexity levels?
2. **Visual faithfulness:** Do answer-token-specific attention and Grad-CAM-style maps identify image regions that causally affect generated answers?
3. **Confidence and grounding:** How are answer correctness, length-normalised sequence confidence, and visual faithfulness related?

### Model Conditions

- Unadapted `google/paligemma-3b-pt-224` pretraining-checkpoint control on the paired image (not described as a zero-shot instruction baseline)
- QLoRA/LoRA fine-tuned PaliGemma on the paired image
- Separately trained constant-image adapter, trained and evaluated with the same neutral image (question-only/question-prior control)
- Fine-tuned PaliGemma with a constant image only at evaluation time (test-time image-ablation diagnostic)
- Fine-tuned PaliGemma with deterministically shuffled images (image-dependence control)
- Optional second VLM only after the primary pipeline is complete

### Primary Endpoint

For the saved deterministic answer, calculate mean random-deletion token log probability minus targeted-deletion token log probability at each region fraction, then average across the locked deletion curve. Positive values indicate that targeted deletion removed more support for the answer. Insertion, regenerated-answer changes, grey-mask replication, and stratified benchmark metrics are secondary endpoints.

For each test item:

```text
original image + question
  -> saved answer, token log probabilities, attribution map
  -> delete or insert top-k attribution regions
  -> re-score the same answer under grey and blur treatments
  -> compare with repeated random and least-salient controls
```

All primary decisions—splits, model conditions, attribution targets, perturbation levels, metrics, and exclusions—must be locked before the leakage-safe grouped test split is evaluated.

## 0. Environment and Reproducibility

This notebook deliberately avoids hardcoded Hugging Face or Weights & Biases tokens. Set them outside the notebook if needed:

```bash
export HF_TOKEN=...
export WANDB_API_KEY=...
```

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from typing import Optional
from importlib.metadata import PackageNotFoundError, version as distribution_version
import platform
import json
import hashlib
import math
import os
import random
import statistics
import numpy as np
from PIL import Image, ImageFilter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def installed_version(distribution: str) -> Optional[str]:
    try:
        return distribution_version(distribution)
    except PackageNotFoundError:
        return None

SOFTWARE_VERSIONS = {
    "python": platform.python_version(),
    **{name: installed_version(name) for name in (
        "torch", "transformers", "datasets", "ms-swift", "scikit-learn",
        "nltk", "rouge-score", "sacrebleu",
    )},
}

@dataclass
class StudyConfig:
    model_name: str = "google/paligemma-3b-pt-224"
    model_revision: str = os.getenv(
        "PALIGEMMA_MODEL_REVISION", "35e4f46485b4d07967e7e9935bc3786aad50687c"
    )
    dataset_revision: str = os.getenv(
        "KVASIR_VQA_X1_REVISION", "61e41148c3214bc5140ad0ab4c28520a512e2a73"
    )
    image_dataset_revision: str = os.getenv(
        "KVASIR_VQA_IMAGE_REVISION", "26df9125b98cbad664e5b1801aac1f70a02689e2"
    )
    data_dir: Path = Path("Kvasir-VQA-x1")
    image_dir: Path = Path("Kvasir-VQA-x1/images")
    output_dir: Path = Path("study_outputs/study_1_gi_vqa_faithfulness")
    official_train_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-train-official.jsonl")
    official_test_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-test-official.jsonl")
    train_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-train-grouped-seed42.jsonl")
    dev_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-dev-grouped-seed42.jsonl")
    test_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-test-grouped-seed42.jsonl")
    question_only_train_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-train-question-only-seed42.jsonl")
    question_only_dev_jsonl: Path = Path("Kvasir-VQA-x1/Kvasir-VQA-x1-dev-question-only-seed42.jsonl")
    faithfulness_subset_jsonl: Path = Path("study_outputs/study_1_gi_vqa_faithfulness/faithfulness_subset_seed42.jsonl")
    dev_image_fraction: float = 0.10
    test_image_fraction: float = 0.10
    inference_sample_size: Optional[int] = None  # None means full split
    perturbation_levels: tuple[float, ...] = (0.05, 0.10, 0.20, 0.30)
    attribution_methods: tuple[str, ...] = (
        "decoder_answer_to_image_attention", "answer_conditioned_grad_cam"
    )
    deletion_treatments: tuple[str, ...] = ("gray", "blur")
    random_control_repeats: int = 5
    bootstrap_repeats: int = 2000
    protocol_path: Path = Path("study_outputs/study_1_gi_vqa_faithfulness/locked_protocol.json")
    constant_image_path: Path = Path("study_outputs/study_1_gi_vqa_faithfulness/control_images/neutral_gray_224.png")

cfg = StudyConfig()
cfg.data_dir.mkdir(parents=True, exist_ok=True)
cfg.image_dir.mkdir(parents=True, exist_ok=True)
cfg.output_dir.mkdir(parents=True, exist_ok=True)

print("Study output dir:", cfg.output_dir)
print("The primary grouped test partition is reserved for the locked final evaluation.")
print("HF token available:", bool(os.getenv("HF_TOKEN")))
print("W&B key available:", bool(os.getenv("WANDB_API_KEY")))

## 1. Data Preparation

Cache the two public official splits, audit their reported image overlap, then construct the paper's primary train/development/test partitions from their union grouped by source `img_id`.

**Leakage rule:** no source image may cross the primary partitions. This notebook uses the original image referenced by each `img_id`; any later image augmentation must carry an explicit parent-source ID and inherit that parent's partition. The published official split is retained only for a separately trained, clearly labelled comparability analysis because an independent MediaEval 2025 audit reported extensive image overlap. The grouped test partition is opened only after the protocol is locked.

In [ ]:
# Run once in a fresh environment.
# !pip install datasets tqdm pillow

from datasets import Image as DatasetImage, load_dataset
from tqdm.auto import tqdm

def write_jsonl(split: str, out_path: Path) -> Path:
    ds = load_dataset("SimulaMet/Kvasir-VQA-x1", split=split, revision=cfg.dataset_revision)
    with out_path.open("w", encoding="utf-8") as f:
        for row in ds:
            record = {
                "messages": [
                    {"role": "user", "content": f"<image>{row['question']}"},
                    {"role": "assistant", "content": row["answer"]},
                ],
                "images": [str(cfg.image_dir / f"{row['img_id']}.jpg")],
                "metadata": {
                    "img_id": row["img_id"],
                    "source_img_id": row["img_id"],
                    "official_split": split,
                    "dataset_revision": cfg.dataset_revision,
                    "complexity": row.get("complexity"),
                    "question_class": row.get("question_class", []),
                    "original": row.get("original"),
                    "image_variant": "original",
                },
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    return out_path

def cache_images(required_ids: set[str]) -> None:
    revision_marker = cfg.image_dir / "_source_revision.txt"
    cache_revision_matches = (
        revision_marker.exists()
        and revision_marker.read_text(encoding="utf-8").strip() == cfg.image_dataset_revision
    )
    host = load_dataset(
        "SimulaMet-HOST/Kvasir-VQA", split="raw", revision=cfg.image_dataset_revision
    ).cast_column("image", DatasetImage(decode=False))
    seen = set()
    for idx, row in tqdm(enumerate(host), total=len(host), desc="Caching unique images"):
        img_id = str(row["img_id"])
        if img_id not in required_ids or img_id in seen:
            continue
        seen.add(img_id)
        out_path = cfg.image_dir / f"{img_id}.jpg"
        needs_write = not cache_revision_matches or not out_path.exists()
        if not needs_write:
            try:
                with Image.open(out_path) as cached_image:
                    cached_image.verify()
            except Exception:
                needs_write = True
        if needs_write:
            image_payload = row["image"]
            image_bytes = image_payload.get("bytes")
            if image_bytes is None and image_payload.get("path"):
                image_bytes = Path(image_payload["path"]).read_bytes()
            if image_bytes is None:
                raise ValueError(f"No encoded image bytes for {img_id}")
            out_path.write_bytes(image_bytes)
            with Image.open(out_path) as cached_image:
                cached_image.verify()
        if seen == required_ids:
            break
    missing = required_ids - seen
    if missing:
        raise FileNotFoundError(f"Image source is missing {len(missing)} required img_ids")
    revision_marker.write_text(cfg.image_dataset_revision + "\n", encoding="utf-8")

def source_image_id(record: dict) -> str:
    metadata = record.get("metadata", {})
    return str(metadata.get("source_img_id") or metadata.get("img_id"))

def record_item_id(record: dict) -> str:
    question = record["messages"][0]["content"].replace("<image>", "").strip()
    key = f"{source_image_id(record)}|{question}".encode("utf-8")
    return hashlib.sha256(key).hexdigest()[:20]

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_records(path: Path, records: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

def ensure_constant_image() -> Path:
    cfg.constant_image_path.parent.mkdir(parents=True, exist_ok=True)
    if not cfg.constant_image_path.exists():
        Image.new("RGB", (224, 224), color=(127, 127, 127)).save(cfg.constant_image_path)
    return cfg.constant_image_path

def write_question_only_copy(source_path: Path, output_path: Path) -> Path:
    constant_path = str(ensure_constant_image())
    records = read_records_without_helper(source_path)
    for record in records:
        record["images"] = [constant_path]
        record.setdefault("metadata", {})["image_variant"] = "constant_training_control"
    write_records(output_path, records)
    return output_path

def read_records_without_helper(path: Path) -> list[dict]:
    return [
        json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()
    ]

def canonical_text(text: str) -> str:
    return " ".join(text.lower().split())

def merge_and_deduplicate(paths: tuple[Path, ...]) -> list[dict]:
    merged = {}
    answers_by_question = defaultdict(set)
    metadata_conflict_keys = set()
    input_rows = 0
    exact_duplicates_removed = 0
    for path in paths:
        for record in read_records_without_helper(path):
            input_rows += 1
            question = record["messages"][0]["content"].replace("<image>", "").strip()
            answer = record["messages"][1]["content"]
            question_key = (source_image_id(record), canonical_text(question))
            answer_key = canonical_text(answer)
            answers_by_question[question_key].add(answer_key)
            key = question_key + (answer_key,)
            official_origin = record.get("metadata", {}).get("official_split")
            if key in merged:
                exact_duplicates_removed += 1
                existing_metadata = merged[key].get("metadata", {})
                current_metadata = record.get("metadata", {})
                comparable_fields = ("complexity", "question_class", "original")
                if any(existing_metadata.get(field) != current_metadata.get(field) for field in comparable_fields):
                    metadata_conflict_keys.add(key)
                origins = merged[key]["metadata"].setdefault("official_origins", [])
                if official_origin not in origins:
                    origins.append(official_origin)
                continue
            record.setdefault("metadata", {})["official_origins"] = [official_origin]
            merged[key] = record
    conflicting_questions = {key for key, answers in answers_by_question.items() if len(answers) > 1}
    conflicting_records = [
        record for key, record in merged.items()
        if key[:2] in conflicting_questions or key in metadata_conflict_keys
    ]
    clean_records = [
        record for key, record in merged.items()
        if key[:2] not in conflicting_questions and key not in metadata_conflict_keys
    ]
    write_records(cfg.output_dir / "excluded_annotation_conflicts.jsonl", conflicting_records)
    merge_report = {
        "input_rows": input_rows,
        "exact_duplicates_removed": exact_duplicates_removed,
        "conflicting_image_question_groups_removed": len(conflicting_questions),
        "metadata_conflict_groups_removed": len(metadata_conflict_keys),
        "conflicting_rows_removed": len(conflicting_records),
        "primary_rows_after_exclusions": len(clean_records),
        "primary_source_images_after_exclusions": len({source_image_id(record) for record in clean_records}),
    }
    if cfg.dataset_revision == "61e41148c3214bc5140ad0ab4c28520a512e2a73":
        expected = {
            "input_rows": 159549, "exact_duplicates_removed": 18,
            "conflicting_image_question_groups_removed": 324,
            "metadata_conflict_groups_removed": 9, "conflicting_rows_removed": 657,
            "primary_rows_after_exclusions": 158874,
            "primary_source_images_after_exclusions": 6449,
        }
        for field, value in expected.items():
            assert merge_report[field] == value, f"Pinned merge audit mismatch for {field}"
    (cfg.output_dir / "merge_and_exclusion_report.json").write_text(
        json.dumps(merge_report, indent=2) + "\n", encoding="utf-8"
    )
    print("Merge/exclusion report:", json.dumps(merge_report, indent=2))
    return clean_records

def make_grouped_train_dev_test_split(
    source_paths: tuple[Path, ...], train_path: Path, dev_path: Path, test_path: Path,
    dev_fraction: float, test_fraction: float, seed: int,
) -> tuple[Path, Path, Path]:
    if not 0 < dev_fraction < 1 or not 0 < test_fraction < 1:
        raise ValueError("Development and test fractions must be between 0 and 1")
    if dev_fraction + test_fraction >= 1:
        raise ValueError("Development plus test fraction must be less than 1")
    records = merge_and_deduplicate(source_paths)
    group_ids = sorted({source_image_id(record) for record in records})
    random.Random(seed).shuffle(group_ids)
    n_test = max(1, round(len(group_ids) * test_fraction))
    n_dev = max(1, round(len(group_ids) * dev_fraction))
    test_ids = set(group_ids[:n_test])
    dev_ids = set(group_ids[n_test:n_test + n_dev])
    train_ids = set(group_ids[n_test + n_dev:])
    partitions = {"train": [], "development": [], "test": []}
    for record in records:
        group_id = source_image_id(record)
        partition = "test" if group_id in test_ids else "development" if group_id in dev_ids else "train"
        record.setdefault("metadata", {})["partition"] = partition
        partitions[partition].append(record)
    write_records(train_path, partitions["train"])
    write_records(dev_path, partitions["development"])
    write_records(test_path, partitions["test"])
    split_manifest = {
        "dataset_revision": cfg.dataset_revision, "seed": seed,
        "train_source_img_ids": sorted(train_ids),
        "development_source_img_ids": sorted(dev_ids),
        "test_source_img_ids": sorted(test_ids),
    }
    (cfg.output_dir / "grouped_split_manifest.json").write_text(
        json.dumps(split_manifest, indent=2) + "\n", encoding="utf-8"
    )
    return train_path, dev_path, test_path

if not cfg.official_train_jsonl.exists() or not cfg.official_test_jsonl.exists():
    write_jsonl("train", cfg.official_train_jsonl)
    write_jsonl("test", cfg.official_test_jsonl)

required_image_ids = {
    source_image_id(record)
    for path in (cfg.official_train_jsonl, cfg.official_test_jsonl)
    for record in read_records_without_helper(path)
}
cache_images(required_image_ids)

make_grouped_train_dev_test_split(
    (cfg.official_train_jsonl, cfg.official_test_jsonl),
    cfg.train_jsonl, cfg.dev_jsonl, cfg.test_jsonl,
    cfg.dev_image_fraction, cfg.test_image_fraction, SEED,
)

write_question_only_copy(cfg.train_jsonl, cfg.question_only_train_jsonl)
write_question_only_copy(cfg.dev_jsonl, cfg.question_only_dev_jsonl)

def assert_record_revision(path: Path) -> None:
    revisions = {record.get("metadata", {}).get("dataset_revision") for record in read_records_without_helper(path)}
    if revisions != {cfg.dataset_revision}:
        raise RuntimeError(f"{path} was built from revisions {revisions}, expected {cfg.dataset_revision}")

for path in (
    cfg.official_train_jsonl, cfg.official_test_jsonl, cfg.train_jsonl, cfg.dev_jsonl, cfg.test_jsonl,
):
    assert_record_revision(path)

print("Official train:", cfg.official_train_jsonl)
print("Official test:", cfg.official_test_jsonl)
print("Grouped train:", cfg.train_jsonl)
print("Grouped development:", cfg.dev_jsonl)
print("Grouped test:", cfg.test_jsonl)

## 2. Data Audit

Before training, verify counts, missing images, repeated QA pairs, metadata coverage, and source-image overlap. The overlap checks are hard gates: training must stop if train, development, or test share a source image.

In [ ]:
def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def audit_jsonl(path: Path) -> dict:
    total = 0
    missing_images = []
    source_ids = []
    answer_lengths = []
    qa_pairs = []
    complexities = Counter()
    question_classes = Counter()
    for record in iter_jsonl(path):
        total += 1
        img_path = Path(record["images"][0])
        if not img_path.exists():
            missing_images.append(str(img_path))
        metadata = record.get("metadata", {})
        source_ids.append(source_image_id(record))
        question = record["messages"][0]["content"].replace("<image>", "").strip()
        answer = record["messages"][1]["content"]
        qa_pairs.append((source_image_id(record), normalize_text_for_audit(question), normalize_text_for_audit(answer)))
        complexities[str(metadata.get("complexity"))] += 1
        for label in metadata.get("question_class") or []:
            question_classes[str(label)] += 1
        answer_lengths.append(len(answer.split()))
    return {
        "path": str(path),
        "total": total,
        "unique_source_images": len(set(source_ids)),
        "repeated_exact_qa_pairs": len(qa_pairs) - len(set(qa_pairs)),
        "missing_images": len(missing_images),
        "example_missing_images": missing_images[:5],
        "mean_answer_words": statistics.mean(answer_lengths) if answer_lengths else 0,
        "complexity_counts": dict(sorted(complexities.items())),
        "question_class_counts": dict(question_classes.most_common()),
    }

def normalize_text_for_audit(text: str) -> str:
    return " ".join(text.lower().split())

def split_source_ids(path: Path) -> set[str]:
    return {source_image_id(record) for record in iter_jsonl(path)}

def input_output_keys(path: Path) -> tuple[set[tuple], set[tuple]]:
    image_question = set()
    image_question_answer = set()
    for record in iter_jsonl(path):
        question = record["messages"][0]["content"].replace("<image>", "").strip()
        answer = record["messages"][1]["content"]
        iq = (source_image_id(record), normalize_text_for_audit(question))
        image_question.add(iq)
        image_question_answer.add(iq + (normalize_text_for_audit(answer),))
    return image_question, image_question_answer

for path in [
    cfg.official_train_jsonl, cfg.official_test_jsonl,
    cfg.train_jsonl, cfg.dev_jsonl, cfg.test_jsonl,
]:
    print(json.dumps(audit_jsonl(path), indent=2))

official_train_ids = split_source_ids(cfg.official_train_jsonl)
official_test_ids = split_source_ids(cfg.official_test_jsonl)
official_train_iq, official_train_iqa = input_output_keys(cfg.official_train_jsonl)
official_test_iq, official_test_iqa = input_output_keys(cfg.official_test_jsonl)
official_split_audit = {
    "dataset_revision": cfg.dataset_revision,
    "official_train_rows": sum(1 for _ in iter_jsonl(cfg.official_train_jsonl)),
    "official_test_rows": sum(1 for _ in iter_jsonl(cfg.official_test_jsonl)),
    "image_dataset_revision": cfg.image_dataset_revision,
    "software_versions": SOFTWARE_VERSIONS,
    "official_train_source_images": len(official_train_ids),
    "official_test_source_images": len(official_test_ids),
    "overlapping_source_images": len(official_train_ids & official_test_ids),
    "test_image_overlap_fraction": len(official_train_ids & official_test_ids) / len(official_test_ids),
    "overlapping_image_question_pairs": len(official_train_iq & official_test_iq),
    "overlapping_image_question_answer_triples": len(official_train_iqa & official_test_iqa),
}
if cfg.dataset_revision == "61e41148c3214bc5140ad0ab4c28520a512e2a73":
    expected = {
        "official_train_rows": 143594, "official_test_rows": 15955,
        "official_train_source_images": 6212, "official_test_source_images": 4058,
        "overlapping_source_images": 3821,
        "overlapping_image_question_pairs": 61,
        "overlapping_image_question_answer_triples": 5,
    }
    for key, value in expected.items():
        assert official_split_audit[key] == value, f"Pinned dataset audit mismatch for {key}"
print("Official split audit:", json.dumps(official_split_audit, indent=2))
(cfg.output_dir / "official_split_audit.json").write_text(
    json.dumps(official_split_audit, indent=2) + "\n", encoding="utf-8"
)

IMAGE_MANIFEST_PATH = cfg.output_dir / "image_manifest.jsonl"
image_manifest = []
for image_id in sorted(official_train_ids | official_test_ids):
    image_path = cfg.image_dir / f"{image_id}.jpg"
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    with Image.open(image_path) as image:
        image.verify()
    image_manifest.append({
        "source_img_id": image_id, "path": str(image_path),
        "bytes": image_path.stat().st_size, "sha256": file_sha256(image_path),
        "image_dataset_revision": cfg.image_dataset_revision,
    })
write_records(IMAGE_MANIFEST_PATH, image_manifest)
print("Image manifest:", IMAGE_MANIFEST_PATH, file_sha256(IMAGE_MANIFEST_PATH))

train_ids = split_source_ids(cfg.train_jsonl)
dev_ids = split_source_ids(cfg.dev_jsonl)
test_ids = split_source_ids(cfg.test_jsonl)
assert train_ids.isdisjoint(dev_ids), "Leakage: train and development share source images"
assert train_ids.isdisjoint(test_ids), "Leakage: train and test share source images"
assert dev_ids.isdisjoint(test_ids), "Leakage: development and test share source images"
primary_ids = train_ids | dev_ids | test_ids
official_union_ids = official_train_ids | official_test_ids
assert primary_ids <= official_union_ids, "Grouped split contains an unknown source image"
print("Primary leakage audit passed: grouped train, development, and test images are disjoint.")
print("Source images removed entirely by annotation-conflict exclusions:", len(official_union_ids - primary_ids))

def build_faithfulness_subset(test_path: Path, output_path: Path, seed: int) -> Path:
    # Select one QA per held-out image using metadata only, before inspecting model outputs.
    by_source = defaultdict(list)
    for record in iter_jsonl(test_path):
        by_source[source_image_id(record)].append(record)
    complexity_counts = Counter()
    class_counts = Counter()
    selected = []
    source_order = sorted(
        by_source, key=lambda value: hashlib.sha256(f"{seed}|{value}".encode()).hexdigest()
    )
    for source_id in source_order:
        def selection_score(record: dict) -> tuple:
            metadata = record.get("metadata", {})
            complexity = str(metadata.get("complexity"))
            labels = tuple(sorted(str(label) for label in metadata.get("question_class") or ["unlabelled"]))
            load = complexity_counts[complexity] + sum(class_counts[label] for label in labels) / len(labels)
            tie_break = hashlib.sha256(f"{seed}|{record_item_id(record)}".encode()).hexdigest()
            return load, tie_break
        chosen = min(by_source[source_id], key=selection_score)
        chosen = dict(chosen)
        chosen["item_id"] = record_item_id(chosen)
        chosen.setdefault("metadata", {})["faithfulness_subset"] = True
        selected.append(chosen)
        metadata = chosen["metadata"]
        complexity_counts[str(metadata.get("complexity"))] += 1
        for label in metadata.get("question_class") or ["unlabelled"]:
            class_counts[str(label)] += 1
    write_records(output_path, selected)
    assert len(selected) == len(by_source), "Faithfulness subset must contain one QA per test image"
    return output_path

build_faithfulness_subset(cfg.test_jsonl, cfg.faithfulness_subset_jsonl, SEED)
print("Faithfulness subset:", cfg.faithfulness_subset_jsonl, file_sha256(cfg.faithfulness_subset_jsonl))

sample = next(iter_jsonl(cfg.train_jsonl))
print("\nExample question:", sample["messages"][0]["content"].replace("<image>", ""))
print("Example answer:", sample["messages"][1]["content"])
print("Example image:", sample["images"][0])

## 3. Lock the Evaluation Protocol

Freeze the confirmatory protocol before producing grouped-test predictions. Development analyses may change the protocol, but every change must occur before locking. After locking, deviations are labelled exploratory.

In [ ]:
STUDY_PROTOCOL = {
    "seed": SEED,
    "dataset": "SimulaMet/Kvasir-VQA-x1",
    "base_model": cfg.model_name,
    "base_model_revision": cfg.model_revision,
    "dataset_revision": cfg.dataset_revision,
    "image_dataset_revision": cfg.image_dataset_revision,
    "software_versions": SOFTWARE_VERSIONS,
    "grouped_split_manifest_sha256": file_sha256(cfg.output_dir / "grouped_split_manifest.json"),
    "image_manifest_sha256": file_sha256(IMAGE_MANIFEST_PATH),
    "faithfulness_subset_sha256": file_sha256(cfg.faithfulness_subset_jsonl),
    "primary_split_file_sha256": {
        "train": file_sha256(cfg.train_jsonl),
        "development": file_sha256(cfg.dev_jsonl),
        "test": file_sha256(cfg.test_jsonl),
    },
    "merge_exclusion_report_sha256": file_sha256(cfg.output_dir / "merge_and_exclusion_report.json"),
    "primary_split": {
        "source": "union of public official train and test",
        "group": "source_img_id",
        "development_fraction": cfg.dev_image_fraction,
        "test_fraction": cfg.test_image_fraction,
        "conflicting_image_question_policy": "exclude all conflicting references",
    },
    "secondary_split": "published official protocol, separately trained checkpoint, leakage-labelled",
    "model_conditions": [
        "unadapted_pt_paired", "finetuned_paired",
        "question_only_constant_image", "finetuned_constant_image_ablation",
        "finetuned_shuffled_image",
    ],
    "primary_model_condition": "finetuned_paired",
    "training": {
        "base_model": cfg.model_name, "base_model_revision": cfg.model_revision,
        "entrypoint": "gi_vqa.training", "template": "gi_vqa_paligemma_v1",
        "token_type_correction": "ms-swift-3.7.0-paligemma-prefix-boundary-v1",
        "train_type": "lora",
        "quantisation": "bnb-nf4-double-quant", "lora_rank": 16, "lora_alpha": 32,
        "learning_rate": 2e-5, "epochs": 1, "primary_seed": SEED,
        "training_seed_sensitivity": [43, 44],
    },
    "decoding": {"max_tokens": 64, "temperature": 0, "seed": SEED, "logprobs": True, "top_logprobs": 1},
    "answer_metrics": ["exact_match", "bleu", "rouge1", "rouge2", "rougeL", "meteor"],
    "strata": ["complexity", "question_class"],
    "attribution_methods": list(cfg.attribution_methods),
    "primary_attribution_method": "answer_conditioned_grad_cam",
    "attribution_target": "mean teacher-forced token log probability of the saved deterministic answer",
    "faithfulness_sampling": "one metadata-balanced QA per grouped-test source image",
    "perturbation_levels": list(cfg.perturbation_levels),
    "deletion_treatments": list(cfg.deletion_treatments),
    "primary_deletion_treatment": "blur",
    "random_control_repeats": cfg.random_control_repeats,
    "controls": ["random_region", "least_salient", "random_map"],
    "primary_endpoint": {
        "name": "fixed-answer targeted deletion effect",
        "per_level": "mean random fixed-answer logprob minus targeted fixed-answer logprob",
        "summary": "mean effect across locked deletion levels; positive favours faithful attribution",
    },
    "secondary_endpoints": [
        "fixed-answer insertion AUC", "regenerated answer flip and correctness change",
        "least-salient effect", "grey-mask replication",
        "category_metrics", "complexity_metrics",
    ],
    "calibration": "logistic mapping from mean sequence logprob to exact-match correctness, fitted on grouped development only",
    "exclusions": [
        "conflicting image-question references removed before splitting",
        "non-finite or constant attribution maps excluded and reported without replacement",
    ],
    "bootstrap_repeats": cfg.bootstrap_repeats,
}

LOCK_PROTOCOL = False  # Set True once development decisions are final.
if cfg.protocol_path.exists():
    locked_protocol = json.loads(cfg.protocol_path.read_text(encoding="utf-8"))
    assert locked_protocol == STUDY_PROTOCOL, "Current settings differ from the locked protocol"
    print("Protocol verified:", cfg.protocol_path)
elif LOCK_PROTOCOL:
    revisions = {
        "KVASIR_VQA_X1_REVISION": cfg.dataset_revision,
        "KVASIR_VQA_IMAGE_REVISION": cfg.image_dataset_revision,
        "PALIGEMMA_MODEL_REVISION": cfg.model_revision,
    }
    unpinned = [name for name, value in revisions.items() if not value or value in {"main", "master"}]
    if unpinned:
        raise ValueError(f"Pin immutable revisions before locking: {unpinned}")
    cfg.protocol_path.write_text(json.dumps(STUDY_PROTOCOL, indent=2) + "\n", encoding="utf-8")
    print("Protocol locked:", cfg.protocol_path)
else:
    print(json.dumps(STUDY_PROTOCOL, indent=2))
    print("Development mode: protocol is not yet locked.")

## 4. Matched Development Fine-Tuning

Train the paired-image and question-only adapters with matched settings on their grouped training partitions. Use only the disjoint grouped development partition for checkpoint and hyperparameter selection. The reported primary checkpoint must never train on grouped development or grouped test data.

Existing adapters trained with the published official split cannot be reused as the primary leakage-safe model; train a new checkpoint from the grouped training partition and record the split-manifest hash.

A secondary official-protocol replication requires a separate checkpoint trained only on the published official train split and evaluated only on the published official test split. Label those results as image-overlapping comparability results, not unseen-image generalisation. Record exactly which training protocol produced every checkpoint.

Seed 42 is the primary run. If compute permits, repeat the paired model with seeds 43 and 44 as a model-level sensitivity analysis; label a single-seed result as a pilot. Set `DRY_RUN = False` only when the environment has the required GPU, packages, and credentials.

Training must use the repository's `python -m gi_vqa.training` entrypoint. The one-item contract identified a real off-by-one in the built-in ms-swift 3.7.0 PaliGemma training `token_type_ids`; raw `swift sft` is therefore not an approved Study 1 path. The project entrypoint forces the versioned corrected template that the contract compares with the direct processor.

In [ ]:
# Run once in a GPU environment.
# !pip install -e "./gi_vqa_research[gpu]"

import shlex
import subprocess
import sys

TRAINING_CONDITION = "paired"  # paired or question_only; train both with matched settings.
TRAINING_SEED = int(os.getenv("STUDY1_TRAINING_SEED", str(SEED)))
if TRAINING_CONDITION == "paired":
    training_dataset = cfg.train_jsonl
    validation_dataset = cfg.dev_jsonl
elif TRAINING_CONDITION == "question_only":
    training_dataset = cfg.question_only_train_jsonl
    validation_dataset = cfg.question_only_dev_jsonl
else:
    raise ValueError("TRAINING_CONDITION must be paired or question_only")

RUN_ID = datetime.now().strftime("%y%m%d-%H%M") + f"-s{TRAINING_SEED}"
HUB_MODEL_ID = f"Kvasir-VQA-x1-study1-{TRAINING_CONDITION}-paligemma-lora-{RUN_ID}"
TRAIN_OUTPUT_DIR = cfg.output_dir / "training" / HUB_MODEL_ID

training_cmd = [
    sys.executable, "-m", "gi_vqa.training",
    "--dataset", str(training_dataset),
    "--val_dataset", str(validation_dataset),
    "--model", cfg.model_name,
    "--model_revision", cfg.model_revision,
    "--max_length", "512",
    "--train_type", "lora",
    "--torch_dtype", "float16",
    "--quant_method", "bnb",
    "--quant_bits", "4",
    "--bnb_4bit_compute_dtype", "float16",
    "--bnb_4bit_quant_type", "nf4",
    "--bnb_4bit_use_double_quant", "true",
    "--num_train_epochs", "1",
    "--per_device_train_batch_size", "4",
    "--per_device_eval_batch_size", "4",
    "--gradient_accumulation_steps", "4",
    "--learning_rate", "2e-5",
    "--lr_scheduler_type", "linear",
    "--warmup_ratio", "0.03",
    "--weight_decay", "0.01",
    "--lora_rank", "16",
    "--lora_alpha", "32",
    "--freeze_vit", "true",
    "--gradient_checkpointing", "true",
    "--load_best_model_at_end", "true",
    "--metric_for_best_model", "eval_loss",
    "--greater_is_better", "false",
    "--save_steps", "1000",
    "--save_total_limit", "2",
    "--logging_steps", "20",
    "--output_dir", str(TRAIN_OUTPUT_DIR),
    "--report_to", "wandb" if os.getenv("WANDB_API_KEY") else "none",
    "--dataloader_num_workers", "2",
    "--dataset_num_proc", "2",
    "--seed", str(TRAINING_SEED),
    "--data_seed", str(TRAINING_SEED),
]

training_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "training_condition": TRAINING_CONDITION,
    "base_model": cfg.model_name,
    "base_model_revision": cfg.model_revision,
    "dataset_revision": cfg.dataset_revision,
    "training_dataset": str(training_dataset),
    "validation_dataset": str(validation_dataset),
    "seed": TRAINING_SEED,
    "software_versions": SOFTWARE_VERSIONS,
    "training_entrypoint": "gi_vqa.training",
    "training_template": "gi_vqa_paligemma_v1",
    "token_type_correction": "ms-swift-3.7.0-paligemma-prefix-boundary-v1",
    "command_without_credentials": training_cmd,
}
TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(TRAIN_OUTPUT_DIR / "training_manifest.json").write_text(
    json.dumps(training_manifest, indent=2) + "\n", encoding="utf-8"
)

if os.getenv("HF_TOKEN"):
    training_cmd += ["--use_hf", "true", "--push_to_hub", "true", "--hub_token", os.getenv("HF_TOKEN"), "--hub_model_id", HUB_MODEL_ID]

print("Training command:\n")
display_cmd = ["***REDACTED***" if part == os.getenv("HF_TOKEN") and part else part for part in training_cmd]
print(" ".join(shlex.quote(part) for part in display_cmd))

DRY_RUN = True
if not DRY_RUN:
    subprocess.run(training_cmd, check=True)

## 5. Deterministic Inference and Shortcut Controls

Run identical deterministic decoding for the unadapted PT, fine-tuned, question-only, constant-image-ablation, and shuffled-image conditions. Save immutable JSONL rows containing the exact evaluated image, generated-token log probabilities, length-normalised sequence confidence, and complete stratification metadata.

Use `development` while building the pipeline. `grouped_test` is permitted only after the protocol is locked, must use every leakage-safe held-out item, and must not be used for model or threshold selection. The published official split requires its own separately trained replication checkpoint and output namespace.

In [ ]:
# Run in a GPU environment after training.
# !pip install ms-swift

PREDICTIONS_DIR = cfg.output_dir / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_DIR = cfg.output_dir / "control_images"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_PATH = os.getenv("PALIGEMMA_ADAPTER_PATH", "")  # local checkpoint or HF adapter repo
ADAPTER_REVISION = os.getenv("PALIGEMMA_ADAPTER_REVISION", "")
QUESTION_ONLY_ADAPTER_PATH = os.getenv("PALIGEMMA_QUESTION_ONLY_ADAPTER_PATH", "")
QUESTION_ONLY_ADAPTER_REVISION = os.getenv("PALIGEMMA_QUESTION_ONLY_ADAPTER_REVISION", "")

MODEL_CONDITIONS = [
    {"name": "unadapted_pt_paired", "adapter_path": None, "adapter_revision": None, "image_mode": "paired", "requires_adapter": False},
    {"name": "finetuned_paired", "adapter_path": ADAPTER_PATH, "adapter_revision": ADAPTER_REVISION, "image_mode": "paired", "requires_adapter": True},
    {"name": "question_only_constant_image", "adapter_path": QUESTION_ONLY_ADAPTER_PATH, "adapter_revision": QUESTION_ONLY_ADAPTER_REVISION, "image_mode": "constant", "requires_adapter": True},
    {"name": "finetuned_constant_image_ablation", "adapter_path": ADAPTER_PATH, "adapter_revision": ADAPTER_REVISION, "image_mode": "constant", "requires_adapter": True},
    {"name": "finetuned_shuffled_image", "adapter_path": ADAPTER_PATH, "adapter_revision": ADAPTER_REVISION, "image_mode": "shuffled", "requires_adapter": True},
]

def load_eval_records(path: Path, sample_size: Optional[int] = None, seed: int = SEED):
    rows = list(iter_jsonl(path))
    if sample_size is not None:
        rows = random.Random(seed).sample(rows, min(sample_size, len(rows)))
    return rows

def stable_item_id(record: dict) -> str:
    question = record["messages"][0]["content"].replace("<image>", "").strip()
    key = f"{source_image_id(record)}|{question}".encode("utf-8")
    return hashlib.sha256(key).hexdigest()[:20]

def image_assignment(records: list[dict], mode: str) -> dict[str, str]:
    if mode == "paired":
        return {stable_item_id(record): record["images"][0] for record in records}
    if mode == "constant":
        constant_path = str(ensure_constant_image())
        return {stable_item_id(record): constant_path for record in records}
    if mode == "shuffled":
        by_source = {}
        for record in records:
            by_source.setdefault(source_image_id(record), record["images"][0])
        source_ids = sorted(by_source)
        if len(source_ids) < 2:
            raise ValueError("Shuffled-image control needs at least two source images")
        shifted_ids = source_ids[1:] + source_ids[:1]
        shuffled_path = {source: by_source[shifted] for source, shifted in zip(source_ids, shifted_ids)}
        return {stable_item_id(record): shuffled_path[source_image_id(record)] for record in records}
    raise ValueError(f"Unknown image mode: {mode}")

def token_logprobs_from_choice(choice) -> list[float]:
    logprobs = getattr(choice, "logprobs", None)
    content = getattr(logprobs, "content", None)
    if content is None and isinstance(logprobs, dict):
        content = logprobs.get("content")
    values = []
    for token_info in content or []:
        value = token_info.get("logprob") if isinstance(token_info, dict) else getattr(token_info, "logprob", None)
        if value is not None and math.isfinite(float(value)):
            values.append(float(value))
    return values

def sequence_confidence(token_logprobs: list[float]) -> Optional[float]:
    # Geometric mean token probability avoids systematically penalising longer answers.
    return math.exp(statistics.mean(token_logprobs)) if token_logprobs else None

EVALUATION_SPLIT = "development"  # Change to grouped_test only after protocol lock.
if EVALUATION_SPLIT == "development":
    evaluation_path = cfg.dev_jsonl
elif EVALUATION_SPLIT == "grouped_test":
    if not cfg.protocol_path.exists():
        raise RuntimeError("Lock the protocol before grouped-test inference")
    if json.loads(cfg.protocol_path.read_text(encoding="utf-8")) != STUDY_PROTOCOL:
        raise RuntimeError("Current settings differ from the locked grouped-test protocol")
    if cfg.inference_sample_size is not None:
        raise ValueError("Grouped-test inference must use the complete held-out split")
    evaluation_path = cfg.test_jsonl
else:
    raise ValueError("EVALUATION_SPLIT must be development or grouped_test")

eval_records = load_eval_records(evaluation_path, cfg.inference_sample_size)
print("Evaluation split:", EVALUATION_SPLIT)
print("Evaluation records:", len(eval_records))

# Skeleton for real inference. Keep this cell explicit so model-loading failures are easy to debug.
RUN_INFERENCE = False

if RUN_INFERENCE:
    from swift.llm import PtEngine, RequestConfig, InferRequest
    import torch
    import gc

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    gc.collect()
    torch.cuda.empty_cache()

    request_config = RequestConfig(
        max_tokens=64, temperature=0, seed=SEED, logprobs=True, top_logprobs=1
    )

    for condition in MODEL_CONDITIONS:
        if condition["requires_adapter"] and not condition["adapter_path"]:
            raise ValueError(f"Missing adapter path for {condition['name']}")
        if EVALUATION_SPLIT == "grouped_test" and condition["requires_adapter"] and not condition["adapter_revision"]:
            raise ValueError(f"Declare an immutable adapter revision/fingerprint for {condition['name']}")
        engine_kwargs = {
            "model_id_or_path": cfg.model_name,
            "max_batch_size": 2,
            "use_hf": True,
            "model_type": "paligemma",
            "revision": cfg.model_revision,
        }
        if condition["adapter_path"]:
            engine_kwargs["adapters"] = [condition["adapter_path"]]
        engine = PtEngine(**engine_kwargs)
        assigned_images = image_assignment(eval_records, condition["image_mode"])
        output_path = PREDICTIONS_DIR / f"{EVALUATION_SPLIT}__{condition['name']}.jsonl"

        with output_path.open("w", encoding="utf-8") as f:
            for start in range(0, len(eval_records), 16):
                batch = eval_records[start:start + 16]
                requests = []
                for record in batch:
                    question = record["messages"][0]["content"].replace("<image>", "").strip()
                    evaluated_image = assigned_images[stable_item_id(record)]
                    requests.append(InferRequest(
                        messages=[{"role": "user", "content": f"<image>{question}"}],
                        images=[evaluated_image],
                    ))
                responses = engine.infer(requests, request_config)
                for record, response in zip(batch, responses):
                    choice = response.choices[0]
                    question = record["messages"][0]["content"].replace("<image>", "").strip()
                    generated_logprobs = token_logprobs_from_choice(choice)
                    confidence = sequence_confidence(generated_logprobs)
                    if confidence is None:
                        raise RuntimeError("Inference backend returned no generated-token log probabilities")
                    out = {
                        "item_id": stable_item_id(record),
                        "condition": condition["name"],
                        "model_name": cfg.model_name,
                        "model_revision": cfg.model_revision,
                        "software_versions": SOFTWARE_VERSIONS,
                        "adapter_path": condition["adapter_path"],
                        "adapter_revision": condition["adapter_revision"],
                        "evaluation_split": EVALUATION_SPLIT,
                        "question": question,
                        "reference": record["messages"][1]["content"],
                        "prediction": choice.message.content.strip(),
                        "source_image": record["images"][0],
                        "evaluated_image": assigned_images[stable_item_id(record)],
                        "token_logprobs": generated_logprobs,
                        "mean_sequence_logprob": statistics.mean(generated_logprobs),
                        "sequence_confidence": confidence,
                        "metadata": record.get("metadata", {}),
                    }
                    f.write(json.dumps(out, ensure_ascii=False) + "\n")
                print(condition["name"], f"{min(start + 16, len(eval_records))}/{len(eval_records)}")
        print("Saved:", output_path)
        del engine
        gc.collect()
        torch.cuda.empty_cache()

## 6. Answer Evaluation and Stratification

Calculate the required benchmark metrics with a pinned implementation and save per-item scores so model conditions can be compared with paired statistics. Report aggregate results and results by complexity and multi-label question class.

Lexical similarity is not equivalent to medical correctness. Any expert correctness/relevance review must use a predefined stratified subset and a separate documented rubric. Match the official scoring implementation before making leaderboard comparisons.

In [ ]:
import re
import unicodedata

# !pip install nltk rouge-score sacrebleu
# Run once if METEOR resources are absent: nltk.download("wordnet"); nltk.download("omw-1.4")
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
import sacrebleu

def normalize_answer(text: str) -> str:
    # Conservative medical-VQA normalization: preserve decimals, units, signs, and punctuation semantics.
    text = unicodedata.normalize("NFKC", text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

assert normalize_answer("  2.5   cm " ) == "2.5 cm"
assert normalize_answer("10%") == "10%"

def exact_match(reference: str, prediction: str) -> bool:
    return normalize_answer(reference) == normalize_answer(prediction)

ROUGE_SCORER = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
BLEU_SMOOTHING = SmoothingFunction().method1
PER_ITEM_METRICS = ("exact_match", "bleu", "rouge1", "rouge2", "rougeL", "meteor")

def score_answer(reference: str, prediction: str) -> dict[str, float]:
    reference_tokens = normalize_answer(reference).split()
    prediction_tokens = normalize_answer(prediction).split()
    rouge = ROUGE_SCORER.score(reference, prediction)
    if reference_tokens and prediction_tokens:
        bleu = sentence_bleu(
            [reference_tokens], prediction_tokens, weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=BLEU_SMOOTHING,
        )
        meteor = meteor_score([reference_tokens], prediction_tokens)
    else:
        bleu = 0.0
        meteor = 0.0
    return {
        "exact_match": float(exact_match(reference, prediction)),
        "bleu": float(bleu),
        "rouge1": float(rouge["rouge1"].fmeasure),
        "rouge2": float(rouge["rouge2"].fmeasure),
        "rougeL": float(rouge["rougeL"].fmeasure),
        "meteor": float(meteor),
    }

def score_prediction_rows(rows: list[dict]) -> list[dict]:
    scored = []
    for row in rows:
        enriched = dict(row)
        enriched["metrics"] = score_answer(row["reference"], row["prediction"])
        scored.append(enriched)
    return scored

def aggregate_scored_rows(rows: list[dict]) -> dict:
    if not rows:
        return {"n": 0}
    result = {"n": len(rows)}
    for metric in PER_ITEM_METRICS:
        result[metric] = statistics.mean(row["metrics"][metric] for row in rows)
    result["corpus_bleu"] = sacrebleu.corpus_bleu(
        [row["prediction"] for row in rows],
        [[row["reference"] for row in rows]],
    ).score / 100.0
    return result

def stratified_results(rows: list[dict]) -> dict:
    complexity_groups = defaultdict(list)
    class_groups = defaultdict(list)
    for row in rows:
        metadata = row.get("metadata", {})
        complexity_groups[str(metadata.get("complexity"))].append(row)
        for label in metadata.get("question_class") or ["unlabelled"]:
            class_groups[str(label)].append(row)
    return {
        "overall": aggregate_scored_rows(rows),
        "by_complexity": {key: aggregate_scored_rows(value) for key, value in sorted(complexity_groups.items())},
        "by_question_class": {key: aggregate_scored_rows(value) for key, value in sorted(class_groups.items())},
    }

prediction_files = [
    path for path in sorted(PREDICTIONS_DIR.glob(f"{EVALUATION_SPLIT}__*.jsonl"))
    if not path.stem.endswith("__scored")
]
for predictions_path in prediction_files:
    rows = score_prediction_rows(list(iter_jsonl(predictions_path)))
    scored_path = predictions_path.with_name(predictions_path.stem + "__scored.jsonl")
    write_records(scored_path, rows)
    results = stratified_results(rows)
    results_path = predictions_path.with_name(predictions_path.stem + "__metrics.json")
    results_path.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
    print(predictions_path.name, json.dumps(results["overall"], indent=2))

if not prediction_files:
    print("No prediction files yet. Run deterministic inference first.")

## 7. Confidence and Calibration

Use mean generated-token log probability as the raw sequence score. Fit a one-dimensional logistic calibration mapping on grouped development predictions only, freeze it, and evaluate it on grouped test predictions using Brier score, expected calibration error (ECE), and risk-coverage summaries.

Normalized exact match is the initial binary correctness target. Repeat the analysis using expert medical-correctness labels on the predefined review subset because lexical mismatch can mark clinically acceptable answers as wrong.

In [ ]:
from sklearn.linear_model import LogisticRegression

def calibration_summary(confidences: np.ndarray, correctness: np.ndarray, n_bins: int = 10) -> dict:
    if len(confidences) == 0:
        return {"n": 0}
    if np.any((confidences < 0) | (confidences > 1)):
        raise ValueError("Sequence confidence must lie in [0, 1]")
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.minimum(np.digitize(confidences, bin_edges[1:-1]), n_bins - 1)
    ece = 0.0
    bins = []
    for bin_id in range(n_bins):
        mask = bin_ids == bin_id
        if not mask.any():
            continue
        mean_confidence = float(confidences[mask].mean())
        accuracy = float(correctness[mask].mean())
        weight = float(mask.mean())
        ece += weight * abs(mean_confidence - accuracy)
        bins.append({
            "bin": bin_id, "n": int(mask.sum()),
            "mean_confidence": mean_confidence, "accuracy": accuracy,
        })
    ranked = sorted(zip(confidences.tolist(), correctness.tolist()), reverse=True)
    risk_coverage = {}
    for coverage in (0.25, 0.50, 0.75, 1.00):
        n_keep = max(1, math.ceil(len(ranked) * coverage))
        selected_accuracy = statistics.mean(label for _, label in ranked[:n_keep])
        risk_coverage[str(coverage)] = {"n": n_keep, "accuracy": selected_accuracy, "risk": 1 - selected_accuracy}
    return {
        "n": len(confidences),
        "accuracy": float(correctness.mean()),
        "mean_confidence": float(confidences.mean()),
        "brier_score": float(np.mean((confidences - correctness) ** 2)),
        "ece": float(ece),
        "bins": bins,
        "risk_coverage": risk_coverage,
    }

def fit_and_evaluate_calibration(development_rows: list[dict], test_rows: list[dict]) -> tuple[dict, list[dict]]:
    x_dev = np.asarray([[float(row["mean_sequence_logprob"])] for row in development_rows])
    y_dev = np.asarray([int(row["metrics"]["exact_match"]) for row in development_rows])
    if len(np.unique(y_dev)) < 2:
        raise ValueError("Development correctness labels need both classes for logistic calibration")
    calibrator = LogisticRegression(random_state=SEED).fit(x_dev, y_dev)
    x_test = np.asarray([[float(row["mean_sequence_logprob"])] for row in test_rows])
    y_test = np.asarray([int(row["metrics"]["exact_match"]) for row in test_rows])
    calibrated = calibrator.predict_proba(x_test)[:, 1]
    enriched_rows = []
    for row, probability in zip(test_rows, calibrated):
        enriched = dict(row)
        enriched["calibrated_correctness_probability"] = float(probability)
        enriched_rows.append(enriched)
    summary = calibration_summary(calibrated, y_test)
    summary["development_n"] = len(development_rows)
    summary["logistic_coefficient"] = float(calibrator.coef_[0, 0])
    summary["logistic_intercept"] = float(calibrator.intercept_[0])
    return summary, enriched_rows

CALIBRATION_CONDITION = "finetuned_paired"
development_scored_path = PREDICTIONS_DIR / f"development__{CALIBRATION_CONDITION}__scored.jsonl"
grouped_test_scored_path = PREDICTIONS_DIR / f"grouped_test__{CALIBRATION_CONDITION}__scored.jsonl"
if development_scored_path.exists() and grouped_test_scored_path.exists():
    summary, calibrated_rows = fit_and_evaluate_calibration(
        list(iter_jsonl(development_scored_path)), list(iter_jsonl(grouped_test_scored_path))
    )
    write_records(PREDICTIONS_DIR / "grouped_test__finetuned_paired__calibrated.jsonl", calibrated_rows)
    (PREDICTIONS_DIR / "grouped_test__finetuned_paired__calibration.json").write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8"
    )
    print(json.dumps(summary, indent=2))
else:
    print("Calibration awaits both development and grouped-test scored predictions.")

## 8. Answer-Token-Specific Visual Attribution Maps

Generate attributions only for the locked one-QA-per-image faithfulness subset. Each attribution must target the exact deterministic answer saved during inference: perform a teacher-forced forward pass and use the mean log probability of those answer tokens as the scalar target.

Study 1 methods:

- `decoder_answer_to_image_attention`: decoder attention from saved answer tokens to image tokens, with layers/heads and aggregation rule recorded
- `answer_conditioned_grad_cam`: Grad-CAM-style attribution whose target is the saved answer-token score
- Random-map negative control

Do not reuse the proof-of-concept maps without validation: generic vision-encoder attention or input gradients that do not target the answer cannot support the paper's causal claim. The attribution-loaded model must reproduce the saved prediction before its map is accepted. Save raw patch-grid arrays, not only rendered heatmaps.

The two model-internal functions below are intentional hard gates. Implement and validate them against the pinned PaliGemma stack before enabling the GPU switches; the surrounding subset, provenance, resume, perturbation, and summarisation workflow is already wired to those interfaces.

In [ ]:
ATTRIBUTION_DIR = cfg.output_dir / "attribution_maps"
ATTRIBUTION_DIR.mkdir(parents=True, exist_ok=True)

def validate_saliency_map(saliency: np.ndarray) -> np.ndarray:
    saliency = np.asarray(saliency, dtype=np.float32)
    if saliency.ndim != 2 or min(saliency.shape) < 2:
        raise ValueError(f"Expected a 2D patch-grid map, received {saliency.shape}")
    if not np.isfinite(saliency).all():
        raise ValueError("Attribution map contains non-finite values")
    if float(saliency.max() - saliency.min()) <= 1e-12:
        raise ValueError("Attribution map is constant")
    saliency = saliency - saliency.min()
    return saliency / saliency.max()

def attribution_path(item_id: str, method: str) -> Path:
    if method not in cfg.attribution_methods and method != "random_map":
        raise ValueError(f"Unregistered attribution method: {method}")
    return ATTRIBUTION_DIR / f"{item_id}__{method}.npz"

def attribution_json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Unsupported attribution metadata type: {type(value).__name__}")

def save_attribution(item_id: str, method: str, saliency: np.ndarray, metadata: dict) -> Path:
    raw_saliency = np.asarray(saliency, dtype=np.float32)
    normalized_saliency = validate_saliency_map(raw_saliency)
    required = {
        "model_revision", "adapter_revision", "target_text", "target_token_ids",
        "image_token_indices", "patch_grid_shape", "preprocessing", "target_mean_logprob",
        "processed_image_path", "processed_image_sha256", "aggregation",
    }
    missing = required - set(metadata)
    if missing:
        raise ValueError(f"Attribution metadata missing: {sorted(missing)}")
    if tuple(metadata["patch_grid_shape"]) != raw_saliency.shape:
        raise ValueError("Recorded patch-grid shape does not match the attribution array")
    processed_image_path = Path(metadata["processed_image_path"])
    if not processed_image_path.exists() or file_sha256(processed_image_path) != metadata["processed_image_sha256"]:
        raise ValueError("Processed image is missing or differs from its recorded hash")
    path = attribution_path(item_id, method)
    np.savez_compressed(
        path, raw_saliency=raw_saliency, normalized_saliency=normalized_saliency,
        metadata=json.dumps(metadata, sort_keys=True, default=attribution_json_default),
    )
    return path

def teacher_forced_answer_score(model_bundle, image: Image.Image, question: str, target_text: str) -> float:
    """Return mean target-token log p(target_text | image, question). Implement for the pinned model stack."""
    raise NotImplementedError("Implement with the attribution-loaded PaliGemma model before GPU execution")

def compute_answer_conditioned_attribution(
    model_bundle, image: Image.Image, question: str, target_text: str, method: str
) -> tuple[np.ndarray, dict]:
    """Return a raw patch-grid attribution and complete provenance metadata."""
    raise NotImplementedError("Implement and validate answer-conditioned attention/Grad-CAM extraction")

def bundle_callable(model_bundle, name: str):
    function = model_bundle.get(name) if isinstance(model_bundle, dict) else getattr(model_bundle, name, None)
    if not callable(function):
        raise TypeError(f"The attribution model bundle must provide a callable {name!r}")
    return function

def prediction_index(path: Path) -> dict[str, dict]:
    if not path.exists():
        raise FileNotFoundError(path)
    rows = list(iter_jsonl(path))
    indexed = {row["item_id"]: row for row in rows}
    if len(indexed) != len(rows):
        raise ValueError(f"Duplicate item IDs in {path}")
    return indexed

def run_locked_attribution_subset(model_bundle, predictions_path: Path) -> Path:
    predictions = prediction_index(predictions_path)
    manifest = []
    for record in iter_jsonl(cfg.faithfulness_subset_jsonl):
        item_id = stable_item_id(record)
        if item_id not in predictions:
            raise KeyError(f"Faithfulness item {item_id} has no saved prediction")
        prediction = predictions[item_id]
        question = prediction["question"]
        target_text = prediction["prediction"]
        image_path = Path(prediction["evaluated_image"])
        with Image.open(image_path) as opened_image:
            image = opened_image.convert("RGB")
            reproduced = bundle_callable(model_bundle, "generate_answer")(image.copy(), question)
            if reproduced.strip() != target_text.strip():
                raise RuntimeError(f"Attribution model did not reproduce saved answer for {item_id}")
            target_score = teacher_forced_answer_score(model_bundle, image.copy(), question, target_text)
            for method in cfg.attribution_methods:
                output_path = attribution_path(item_id, method)
                if not output_path.exists():
                    saliency, metadata = compute_answer_conditioned_attribution(
                        model_bundle, image.copy(), question, target_text, method
                    )
                    metadata = dict(metadata)
                    metadata.update({
                        "item_id": item_id, "method": method,
                        "model_revision": prediction["model_revision"],
                        "adapter_revision": prediction["adapter_revision"],
                        "target_text": target_text,
                        "target_mean_logprob": float(target_score),
                        "source_image_path": str(image_path),
                        "source_image_sha256": file_sha256(image_path),
                        "saved_prediction_reproduced": True,
                    })
                    save_attribution(item_id, method, saliency, metadata)
                else:
                    with np.load(output_path, allow_pickle=False) as existing_payload:
                        existing_metadata = json.loads(existing_payload["metadata"].item())
                    expected_metadata = {
                        "item_id": item_id, "method": method, "target_text": target_text,
                        "model_revision": prediction["model_revision"],
                        "adapter_revision": prediction["adapter_revision"],
                        "source_image_sha256": file_sha256(image_path),
                    }
                    if any(existing_metadata.get(key) != value for key, value in expected_metadata.items()):
                        raise RuntimeError(f"Saved attribution provenance changed for {item_id}/{method}")
                manifest.append({
                    "item_id": item_id, "method": method, "path": str(output_path),
                    "sha256": file_sha256(output_path),
                })
    manifest_path = ATTRIBUTION_DIR / "attribution_manifest.jsonl"
    write_records(manifest_path, manifest)
    return manifest_path

ATTRIBUTION_MODEL_BUNDLE = None  # Supply the pinned adapter/model implementation here.
RUN_ATTRIBUTION = False
if RUN_ATTRIBUTION:
    if ATTRIBUTION_MODEL_BUNDLE is None:
        raise RuntimeError("Assign ATTRIBUTION_MODEL_BUNDLE before enabling attribution")
    manifest_path = run_locked_attribution_subset(
        ATTRIBUTION_MODEL_BUNDLE,
        PREDICTIONS_DIR / "grouped_test__finetuned_paired.jsonl",
    )
    print("Attribution manifest:", manifest_path)

print("Attribution directory:", ATTRIBUTION_DIR)

## 9. Fixed-Answer Faithfulness Perturbation Tests

This is the main study contribution.

For each locked-subset item, explain the saved deterministic prediction and keep that target answer fixed across interventions:

1. Score the saved answer under the original image using mean teacher-forced token log probability.
2. Delete top-k attributed patches and score the same saved answer.
3. Repeat with the same number and shape of random and least-salient patches.
4. Average random repeats within each item and deletion level.
5. Calculate `random fixed-answer logprob - targeted fixed-answer logprob`; positive values mean targeted deletion removed more support.
6. Average the effect across locked deletion levels for the primary AOPC-style endpoint.

Blur is the primary deletion treatment and grey replacement is a replication. Insertion from a blurred baseline, regenerated-answer changes, least-salient deletion, and random-map attribution are secondary controls. Fixed-answer scoring avoids treating harmless paraphrases as answer flips and avoids a correctness floor for originally incorrect predictions.

In [ ]:
# Patch-level image interventions. Use the same processed-image coordinates as attribution extraction.
# !pip install pillow numpy

PERTURBED_DIR = cfg.output_dir / "perturbed_images"
PERTURBED_DIR.mkdir(parents=True, exist_ok=True)

def stable_seed(*parts) -> int:
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big") % (2**32)

def make_patch_mask(saliency: np.ndarray, fraction: float, mode: str, seed: int) -> np.ndarray:
    saliency = validate_saliency_map(saliency)
    if not 0 < fraction <= 1:
        raise ValueError("fraction must be in (0, 1]")
    flat = saliency.ravel()
    n_mask = max(1, round(len(flat) * fraction))
    if mode == "targeted":
        indices = np.argsort(flat, kind="stable")[-n_mask:]
    elif mode == "least_salient":
        indices = np.argsort(flat, kind="stable")[:n_mask]
    elif mode == "random":
        rng = np.random.default_rng(seed)
        indices = rng.choice(len(flat), size=n_mask, replace=False)
    else:
        raise ValueError(f"Unknown deletion mode: {mode}")
    mask = np.zeros_like(flat, dtype=bool)
    mask[indices] = True
    patch_mask = mask.reshape(saliency.shape)
    assert int(patch_mask.sum()) == n_mask
    return patch_mask

def resize_patch_mask(patch_mask: np.ndarray, image_size: tuple[int, int]) -> np.ndarray:
    mask_image = Image.fromarray((patch_mask.astype(np.uint8) * 255), mode="L")
    mask_image = mask_image.resize(image_size, resample=Image.Resampling.NEAREST)
    return np.asarray(mask_image) > 0

def apply_deletion(
    processed_image: Image.Image, saliency: np.ndarray, fraction: float, mode: str,
    treatment: str, seed: int, blur_radius: float = 12.0,
) -> Image.Image:
    image = processed_image.convert("RGB")
    image_arr = np.asarray(image).copy()
    patch_mask = make_patch_mask(saliency, fraction=fraction, mode=mode, seed=seed)
    mask = resize_patch_mask(patch_mask, image.size)
    if treatment == "gray":
        replacement = np.full_like(image_arr, 127)
    elif treatment == "blur":
        replacement = np.asarray(image.filter(ImageFilter.GaussianBlur(radius=blur_radius)))
    else:
        raise ValueError(f"Unknown deletion treatment: {treatment}")
    image_arr[mask] = replacement[mask]
    return Image.fromarray(image_arr)

def apply_insertion(
    processed_image: Image.Image, saliency: np.ndarray, fraction: float, mode: str, seed: int,
    blur_radius: float = 12.0,
) -> Image.Image:
    image = processed_image.convert("RGB")
    original = np.asarray(image)
    result = np.asarray(image.filter(ImageFilter.GaussianBlur(radius=blur_radius))).copy()
    patch_mask = make_patch_mask(saliency, fraction=fraction, mode=mode, seed=seed)
    mask = resize_patch_mask(patch_mask, image.size)
    result[mask] = original[mask]
    return Image.fromarray(result)

def random_map_like(saliency: np.ndarray, item_id: str, repeat: int) -> np.ndarray:
    rng = np.random.default_rng(stable_seed(SEED, item_id, "random_map", repeat))
    return rng.random(np.asarray(saliency).shape, dtype=np.float32)

def intervention_seed(item_id: str, method: str, fraction: float, repeat: int) -> int:
    return stable_seed(SEED, item_id, method, fraction, repeat)

def fixed_answer_deletion_effect(rows: list[dict]) -> dict:
    targeted = [row for row in rows if row["mode"] == "targeted"]
    random_rows = [row for row in rows if row["mode"] == "random"]
    if len(targeted) != 1 or not random_rows:
        raise ValueError("Each item/method/fraction/treatment needs one targeted and repeated random scores")
    targeted_score = float(targeted[0]["fixed_answer_mean_logprob"])
    random_score = statistics.mean(float(row["fixed_answer_mean_logprob"]) for row in random_rows)
    return {
        "targeted_fixed_answer_mean_logprob": targeted_score,
        "random_fixed_answer_mean_logprob": random_score,
        "targeted_deletion_effect": random_score - targeted_score,
    }

def summarise_deletion_effects(rows: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    for row in rows:
        if row.get("intervention") != "deletion" or row["method"] == "random_map":
            continue
        key = (row["item_id"], row["method"], row["treatment"], float(row["fraction"]))
        grouped[key].append(row)
    per_level = []
    for (item_id, method, treatment, fraction), group in sorted(grouped.items()):
        effect = fixed_answer_deletion_effect(group)
        per_level.append({
            "item_id": item_id, "source_img_id": str(group[0]["source_img_id"]),
            "method": method, "treatment": treatment,
            "fraction": fraction, **effect,
        })
    by_item = defaultdict(list)
    for row in per_level:
        by_item[(row["item_id"], row["method"], row["treatment"])].append(row)
    return [
        {
            "item_id": item_id, "source_img_id": item_rows[0]["source_img_id"],
            "method": method, "treatment": treatment,
            "mean_targeted_deletion_effect": statistics.mean(
                row["targeted_deletion_effect"] for row in item_rows
            ),
            "per_level": item_rows,
        }
        for (item_id, method, treatment), item_rows in sorted(by_item.items())
    ]

def load_attribution(item_id: str, method: str) -> tuple[np.ndarray, dict, Path]:
    path = attribution_path(item_id, method)
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as payload:
        saliency = validate_saliency_map(payload["normalized_saliency"])
        metadata = json.loads(payload["metadata"].item())
    if tuple(metadata["patch_grid_shape"]) != saliency.shape:
        raise ValueError(f"Attribution shape mismatch in {path}")
    return saliency, metadata, path

def intervention_id(
    item_id: str, method: str, intervention: str, treatment: str,
    fraction: float, mode: str, repeat: int,
) -> str:
    value = f"{item_id}|{method}|{intervention}|{treatment}|{fraction:.6f}|{mode}|{repeat}"
    return hashlib.sha256(value.encode("utf-8")).hexdigest()[:24]

def run_fixed_answer_perturbations(
    model_bundle, predictions_path: Path, include_insertion: bool = True,
) -> tuple[Path, Path]:
    predictions = prediction_index(predictions_path)
    output_path = cfg.output_dir / "fixed_answer_perturbations.jsonl"
    existing_rows = list(iter_jsonl(output_path)) if output_path.exists() else []
    existing = {row["intervention_id"]: row for row in existing_rows}
    if len(existing) != len(existing_rows):
        raise ValueError("Duplicate intervention IDs in existing perturbation results")

    def score_and_append(
        output_file, prediction: dict, item_id: str, method: str, intervention: str,
        treatment: str, fraction: float, mode: str, repeat: int, image: Image.Image,
        attribution_file: Optional[Path], original_score: float,
    ) -> None:
        run_id = intervention_id(item_id, method, intervention, treatment, fraction, mode, repeat)
        if run_id in existing:
            saved_row = existing[run_id]
            saved_path = Path(saved_row["perturbed_image_path"])
            if not saved_path.exists() or file_sha256(saved_path) != saved_row["perturbed_image_sha256"]:
                raise RuntimeError(f"Existing perturbation artifact failed verification: {run_id}")
            if attribution_file and file_sha256(attribution_file) != saved_row["attribution_sha256"]:
                raise RuntimeError(f"Attribution changed beneath resumed intervention: {run_id}")
            if (
                saved_row["target_text"] != prediction["prediction"]
                or saved_row["model_revision"] != prediction["model_revision"]
                or saved_row["adapter_revision"] != prediction["adapter_revision"]
            ):
                raise RuntimeError(f"Model target changed beneath resumed intervention: {run_id}")
            return
        image_path = PERTURBED_DIR / f"{item_id}__{run_id}.png"
        image.save(image_path, format="PNG")
        fixed_score = teacher_forced_answer_score(
            model_bundle, image, prediction["question"], prediction["prediction"]
        )
        row = {
            "intervention_id": run_id, "item_id": item_id,
            "source_img_id": str(prediction["metadata"]["source_img_id"]),
            "method": method, "intervention": intervention, "treatment": treatment,
            "fraction": float(fraction), "mode": mode, "repeat": repeat,
            "target_text": prediction["prediction"],
            "original_fixed_answer_mean_logprob": float(original_score),
            "fixed_answer_mean_logprob": float(fixed_score),
            "perturbed_image_path": str(image_path),
            "perturbed_image_sha256": file_sha256(image_path),
            "attribution_path": str(attribution_file) if attribution_file else None,
            "attribution_sha256": file_sha256(attribution_file) if attribution_file else None,
            "model_revision": prediction["model_revision"],
            "adapter_revision": prediction["adapter_revision"],
        }
        output_file.write(json.dumps(row, ensure_ascii=False) + "\n")
        output_file.flush()
        existing[run_id] = row

    modes = [("targeted", 0), ("least_salient", 0)] + [
        ("random", repeat) for repeat in range(cfg.random_control_repeats)
    ]
    with output_path.open("a", encoding="utf-8") as output_file:
        for record in iter_jsonl(cfg.faithfulness_subset_jsonl):
            item_id = stable_item_id(record)
            if item_id not in predictions:
                raise KeyError(f"Faithfulness item {item_id} has no saved prediction")
            prediction = predictions[item_id]
            base_saliency = None
            base_processed_image = None
            base_original_score = None
            base_processed_sha256 = None
            for method in cfg.attribution_methods:
                saliency, metadata, attribution_file = load_attribution(item_id, method)
                if metadata["target_text"].strip() != prediction["prediction"].strip():
                    raise RuntimeError(f"Attribution target differs from saved prediction for {item_id}")
                processed_path = Path(metadata["processed_image_path"])
                if file_sha256(processed_path) != metadata["processed_image_sha256"]:
                    raise RuntimeError(f"Processed-image hash mismatch for {item_id}")
                with Image.open(processed_path) as opened_image:
                    processed_image = opened_image.convert("RGB")
                original_score = float(metadata["target_mean_logprob"])
                if base_saliency is None:
                    base_saliency = saliency
                    base_processed_image = processed_image.copy()
                    base_original_score = original_score
                    base_processed_sha256 = metadata["processed_image_sha256"]
                elif (
                    metadata["processed_image_sha256"] != base_processed_sha256
                    or not math.isclose(original_score, base_original_score, rel_tol=0.0, abs_tol=1e-6)
                ):
                    raise RuntimeError(f"Attribution methods used different image/target scores for {item_id}")
                for fraction in cfg.perturbation_levels:
                    for mode, repeat in modes:
                        seed = intervention_seed(item_id, method, fraction, repeat)
                        for treatment in cfg.deletion_treatments:
                            deleted = apply_deletion(
                                processed_image, saliency, fraction, mode, treatment, seed
                            )
                            score_and_append(
                                output_file, prediction, item_id, method, "deletion", treatment,
                                fraction, mode, repeat, deleted, attribution_file, original_score,
                            )
                        if include_insertion:
                            inserted = apply_insertion(
                                processed_image, saliency, fraction, mode, seed
                            )
                            score_and_append(
                                output_file, prediction, item_id, method, "insertion",
                                "blur_baseline", fraction, mode, repeat, inserted,
                                attribution_file, original_score,
                            )
            for repeat in range(cfg.random_control_repeats):
                random_saliency = random_map_like(base_saliency, item_id, repeat)
                for fraction in cfg.perturbation_levels:
                    seed = intervention_seed(item_id, "random_map", fraction, repeat)
                    for treatment in cfg.deletion_treatments:
                        deleted = apply_deletion(
                            base_processed_image, random_saliency, fraction, "targeted", treatment, seed
                        )
                        score_and_append(
                            output_file, prediction, item_id, "random_map", "deletion",
                            treatment, fraction, "targeted", repeat, deleted, None,
                            base_original_score,
                        )
    all_rows = list(iter_jsonl(output_path))
    summary_path = cfg.output_dir / "fixed_answer_deletion_effects_by_item.jsonl"
    write_records(summary_path, summarise_deletion_effects(all_rows))
    return output_path, summary_path

RUN_PERTURBATION_INFERENCE = False
if RUN_PERTURBATION_INFERENCE:
    if ATTRIBUTION_MODEL_BUNDLE is None:
        raise RuntimeError("Assign ATTRIBUTION_MODEL_BUNDLE before perturbation inference")
    perturbation_path, summary_path = run_fixed_answer_perturbations(
        ATTRIBUTION_MODEL_BUNDLE,
        PREDICTIONS_DIR / "grouped_test__finetuned_paired.jsonl",
    )
    print("Perturbation rows:", perturbation_path)
    print("Deletion-effect summary:", summary_path)

print("Perturbation levels:", cfg.perturbation_levels)
print("Random repeats:", cfg.random_control_repeats)
print("Perturbed image directory:", PERTURBED_DIR)

## 10. Paired and Clustered Confidence Intervals

Use paired comparisons by `item_id`. For the full QA benchmark, resample source-image clusters rather than treating many questions about the same image as independent. The locked faithfulness subset contains one QA per image, but it uses the same cluster-bootstrap implementation for consistency. Average random repeats within an item before bootstrapping.

In [ ]:
def cluster_bootstrap_ci(
    rows: list[dict], statistic_fn, cluster_fn, n_boot: int = 2000, ci: float = 0.95, seed: int = SEED
) -> dict:
    if not rows:
        return {"estimate": None, "low": None, "high": None}
    clusters = defaultdict(list)
    for row in rows:
        clusters[str(cluster_fn(row))].append(row)
    cluster_ids = sorted(clusters)
    rng = random.Random(seed)
    estimates = []
    for _ in range(n_boot):
        sampled_ids = [cluster_ids[rng.randrange(len(cluster_ids))] for _ in cluster_ids]
        sample = [row for cluster_id in sampled_ids for row in clusters[cluster_id]]
        estimates.append(float(statistic_fn(sample)))
    estimates.sort()
    alpha = 1 - ci
    low = estimates[int((alpha / 2) * n_boot)]
    high = estimates[int((1 - alpha / 2) * n_boot) - 1]
    return {
        "estimate": float(statistic_fn(rows)), "low": low, "high": high,
        "n_rows": len(rows), "n_clusters": len(cluster_ids), "n_boot": n_boot,
    }

def paired_metric_differences(rows_a: list[dict], rows_b: list[dict], metric: str) -> list[dict]:
    a_by_id = {row["item_id"]: row for row in rows_a}
    b_by_id = {row["item_id"]: row for row in rows_b}
    if set(a_by_id) != set(b_by_id):
        raise ValueError("Paired conditions have different item IDs")
    return [
        {
            "item_id": item_id,
            "source_img_id": source_image_id(a_by_id[item_id]),
            "difference": a_by_id[item_id]["metrics"][metric] - b_by_id[item_id]["metrics"][metric],
        }
        for item_id in sorted(a_by_id)
    ]

# Example for paired condition differences:
# differences = paired_metric_differences(finetuned_rows, unadapted_rows, "rougeL")
# cluster_bootstrap_ci(
#     differences, lambda sample: statistics.mean(row["difference"] for row in sample),
#     lambda row: row["source_img_id"], n_boot=cfg.bootstrap_repeats,
# )

## 11. Operational Failure-Mode Taxonomy

Use this table while manually reviewing examples. The aim is to build a defensible taxonomy, not just cherry-pick interesting images.

| Category | Description |
| --- | --- |
| Correct, high sensitivity | Reference-matching answer with a large targeted-versus-random deletion effect |
| Correct, low sensitivity | Reference-matching answer with little targeted-versus-random effect |
| Incorrect, high sensitivity | Reference-mismatching answer nevertheless depends strongly on attributed evidence |
| Incorrect, low sensitivity | Reference-mismatching answer with little evidence of attributed visual dependence |
| High-confidence incorrect | Incorrect answer assigned a high development-calibrated probability of correctness |
| Image-invariant | Paired, constant-image, or shuffled-image conditions produce the same answer/score |
| Attribution-method disagreement | Answer-conditioned attention and Grad-CAM produce materially different effects or maps |
| Treatment disagreement | Faithfulness conclusion changes between blur and grey deletion |
| Attribution instability | Repeated deterministic extraction fails to reproduce the map |

## 12. Reporting Checklist and Claim Boundaries

- Pinned QA/image dataset revisions, licences, provenance, and cached-image verification
- Reproduced official train/test image overlap and duplicate/conflicting-reference audit
- Versioned leakage-safe split manifest, exclusions, counts, and class/complexity distributions
- Strict separation of primary grouped-split results from the secondary official-protocol replication
- Resolved model, adapter, training, LoRA, quantisation, preprocessing, seed, and decoding settings
- A 20-item development smoke test covering training-load, generation, attribution, all perturbation modes, resume behaviour, and scoring before any full run
- Repeated-run checks for identical deterministic predictions and finite, nonconstant, reproducible attribution maps that change when the question changes for the same image
- Full grouped-test BLEU, ROUGE-1/2/L, METEOR, exact match, counts, and clustered intervals
- Results stratified by complexity and multi-label question class
- Paired comparisons for unadapted, fine-tuned, question-only, constant-ablation, and shuffled-image conditions
- Primary fixed-answer targeted-deletion effect with source-image cluster bootstrap intervals
- Insertion, least-salient, random-map, grey-mask, and regenerated-answer secondary analyses
- Development-fitted calibration evaluated once on grouped test, with Brier, ECE, and risk-coverage
- Predefined operational failure categories and systematic, stratified examples
- Explicit limitations: lexical metrics are proxies, attribution is method-dependent, grouped data remain retrospective, and there is no clinical deployment or clinician-trust claim

## 13. Study 1 Sources and Benchmark Context

- [MediaEval Medico 2025 task repository](https://github.com/simula/MediaEval-Medico-2025) — archived task requirements and submission formats
- [Kvasir-VQA](https://arxiv.org/abs/2409.01437) and [Kvasir-VQA-x1](https://arxiv.org/abs/2506.09958) — dataset lineage and expanded reasoning benchmark
- [HyperKvasir](https://www.nature.com/articles/s41597-020-00622-y) — upstream GI endoscopy data provenance
- [On evaluation metrics for medical applications of artificial intelligence](https://www.nature.com/articles/s41598-022-09954-8) — rationale for reporting multiple, interpretable metrics rather than relying on one score
- [Visual Question Answering benchmark](https://visualqa.org/) — general VQA evaluation context; its multi-reference accuracy is not directly transferable to this single-reference medical dataset
- [ImageCLEFmed MEDVQA-GI 2023](https://www.imageclef.org/2023/medical/vqa) — prior GI-VQA and visual-localisation benchmark context
- [Data Leakage in Kvasir-VQA-x1](https://2025.multimediaeval.com/paper29.pdf) — published image-overlap audit motivating the primary grouped split

Record access dates and immutable revisions in the paper's reproducibility appendix. The 2025 competition has concluded; its repository is used as an archived specification rather than an active deadline.